# CNS2025 Homework 13

Entropy rate of a spike train

This notebook expects `hw13-data.npz` in the same folder.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("hw13-data.npz")
spike_train = data["spike_train"].astype(int)
delta_t = float(data["delta_t"])

total_time = len(spike_train) * delta_t
num_spikes = int(spike_train.sum())
rate = num_spikes / total_time

print(f"Total time: {total_time:.3f} s")
print(f"Number of spikes: {num_spikes}")
print(f"Mean firing rate: {rate:.3f} Hz")


In [ ]:
def entropy_bits(prob):
    prob = prob[prob > 0]
    return -(prob * np.log2(prob)).sum()

def entropy_rate_for_Ts(spike_train, delta_t, Ts_ms):
    Ts_s = Ts_ms / 1000.0
    Ts_bins = int(round(Ts_s / delta_t))
    if Ts_bins < 1 or Ts_bins >= len(spike_train):
        raise ValueError(f"Ts_ms={Ts_ms} ms is invalid for this data.")

    # sliding window over the spike train
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(spike_train, Ts_bins)
    # follow the convention in code13: ignore the very last window
    windows = windows[:-1]

    _, counts = np.unique(windows, axis=0, return_counts=True)
    prob = counts / counts.sum()

    H = entropy_bits(prob)              # bits
    H_rate = H / (Ts_bins * delta_t)    # bits/s
    return H_rate, Ts_bins


In [ ]:
Ts_ms_list = np.arange(5, 101, 5)  # 5,10,...,100 ms
entropy_rates = []
Ts_bins_list = []

for Ts_ms in Ts_ms_list:
    H_rate, Ts_bins = entropy_rate_for_Ts(spike_train, delta_t, Ts_ms)
    entropy_rates.append(H_rate)
    Ts_bins_list.append(Ts_bins)
    print(f"Ts = {Ts_ms:3d} ms ({Ts_bins:4d} bins): H_rate = {H_rate:7.3f} bits/s")

entropy_rates = np.array(entropy_rates)
Ts_bins_list = np.array(Ts_bins_list)


In [ ]:
Ts_s = Ts_ms_list / 1000.0
x = 1.0 / Ts_s        # 1/Ts in Hz
y = entropy_rates     # bits/s

coef = np.polyfit(x, y, 1)
a, b = coef[0], coef[1]

print()
print(f"Linear fit: H_rate ≈ {a:.4f} * (1/Ts) + {b:.4f}")
print(f"Estimated entropy rate at 1/Ts → 0: {b:.4f} bits/s")

x_fit = np.linspace(0, x.max() * 1.05, 200)
y_fit = a * x_fit + b

plt.figure(figsize=(6, 4))
plt.scatter(x, y, label="data")
plt.plot(x_fit, y_fit, label="linear fit")
plt.xlabel("1 / Ts (Hz)")
plt.ylabel("Entropy rate (bits/s)")
plt.title("Entropy rate vs 1/Ts")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# (Optional) information per spike using the extrapolated rate
info_rate_extrap = b          # bits/s
info_per_spike = info_rate_extrap / rate
print(f"Estimated information per spike: {info_per_spike:.3f} bits/spike")
